# 🌳 Árvore Geradora (Spanning Tree)

Uma **árvore geradora** de um grafo conexo $G=(V,E)$ é um subgrafo $T=(V, E_T)$ que é **árvore** e contém **todos os vértices** de $G$. Em particular, $|E_T| = |V|-1$.

Propriedades fundamentais:
- Todo grafo conexo possui pelo menos uma árvore geradora.
- Duas árvores geradoras quaisquer de um mesmo grafo têm o mesmo número de arestas (igual a $|V|-1$).
- Remover qualquer aresta de um circuito reduz o número de arestas e pode quebrar o ciclo; iterando, obtemos uma árvore.

## ✈️ Exemplo (Rotas Aéreas)
Considere um grafo que conecta cidades por rotas. Uma árvore geradora representa um conjunto mínimo de rotas que mantém todas as cidades conectadas (sem ciclos), útil para reduzir custos fixos.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from typing import Iterable, Tuple

def is_spanning_tree(G: nx.Graph, T: nx.Graph) -> bool:
    # Mesmos vértices
    if set(G.nodes()) != set(T.nodes()):
        return False
    # T deve ser conexo e sem ciclos, e ter |V|-1 arestas
    if not nx.is_tree(T):
        return False
    return True

def spanning_tree_via_bfs(G: nx.Graph, s=None) -> nx.Graph:
    # Usa arestas de uma BFS para formar T
    if s is None:
        s = next(iter(G.nodes()))
    T = nx.Graph()
    T.add_nodes_from(G.nodes())
    for u, v in nx.bfs_edges(G, s):
        T.add_edge(u, v)
    # Se grafo não for conexo, adicionar BFS de outros componentes
    for comp in nx.connected_components(G):
        start = next(iter(comp))
        for u, v in nx.bfs_edges(G.subgraph(comp), start):
            T.add_edge(u, v)
    return T

def spanning_tree_by_cycle_removal(G: nx.Graph) -> nx.Graph:
    # Estratégia: copiar G e remover arestas de ciclos até restar uma árvore
    T = G.copy()
    # Enquanto existir ciclo, remove uma aresta do ciclo
    try:
        while True:
            c = nx.find_cycle(T)  # lista de arestas do ciclo
            # remove a primeira aresta do ciclo
            u, v = c[0][:2]
            T.remove_edge(u, v)
            if nx.is_tree(T):
                break
    except nx.exception.NetworkXNoCycle:
        pass
    # Garantir que seja árvore (se conexo e |E|=|V|-1)
    if not nx.is_connected(T):
        raise ValueError('G não é conexo; árvore geradora exige conectividade.')
    if T.number_of_edges() != T.number_of_nodes() - 1:
        raise ValueError('Resultado não tem |V|-1 arestas; verifique o grafo.')
    return T

## 🧪 Exemplo prático

In [ ]:
G = nx.Graph()
G.add_edges_from([(0,1),(1,2),(2,3),(3,4),(4,0),(1,3)])  # contém ciclos
pos = nx.spring_layout(G, seed=10)

T1 = spanning_tree_via_bfs(G)
T2 = spanning_tree_by_cycle_removal(G)
print('T1 é árvore geradora?', is_spanning_tree(G, T1))
print('T2 é árvore geradora?', is_spanning_tree(G, T2))

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=900, font_size=12)
plt.title('Grafo original (com ciclos)')

plt.subplot(1,2,2)
edge_colors = ['crimson' if e in T1.edges() or (e[1],e[0]) in T1.edges() else 'gray' for e in G.edges()]
nx.draw(G, pos, with_labels=True, node_color='lightgreen', node_size=900, font_size=12, edge_color=edge_colors, width=3)
plt.title('Árvore geradora via BFS (arestas em vermelho)')
plt.tight_layout()
plt.show()

## ⏱️ Complexidade
- Obter uma árvore geradora via BFS/DFS custa O(|V|+|E|).
- Remoção iterativa de arestas de ciclos depende do método para detectar ciclos (por ex., find_cycle do NetworkX); na prática, usar BFS/DFS é preferível.

## 🧩 Exercícios
1. Gere três árvores geradoras diferentes do mesmo grafo e compare suas arestas.
2. Implemente uma função que, dado G, retorna uma árvore geradora usando apenas DFS (sem networkx.bfs_edges).
3. Prove que qualquer árvore geradora de um grafo com |V|=n tem exatamente n−1 arestas.
4. Dado um grafo desconexo, adapte o código para retornar uma floresta geradora (uma árvore por componente).